# Spark MERGE does not reconcile a prior read — Fabric Spark, SQL

Companion to `occ.ipynb`. Fabric Spark: Delta is the default, nothing to install, `spark` is already here. Everything below is SQL. **No time travel** — the target is read the ordinary way, exactly as Spark intends.

**Claim.** A pipeline reads the target, decides its work from what it read, then `MERGE`s. If another process changes the target *between the read and the merge*, Spark does not notice: the merge opens a fresh transaction at current HEAD with no link to the earlier read, and commits — silently applying a work list that is already wrong.

delta-rs catches this exact timeline with `DeltaTable(path, version=vB).merge(...)` -> `ConcurrentDeleteRead` (see `occ.ipynb`). Spark exposes no equivalent pin.

In [1]:
# Fabric Spark: `spark` is provided, Delta is the default format. No setup, no install.
spark.sql("DROP TABLE IF EXISTS occ_spark_target")

# All 5 batches as a source view: 20 ids each, filename = provenance (no files needed).
spark.sql("""
    CREATE OR REPLACE TEMP VIEW all_batches AS
    SELECT id,
           id * 10 AS value,
           concat('batch_', lpad(cast(ceil(id / 20.0) AS int), 3, '0'), '.csv') AS filename
    FROM range(1, 101)
""")

# Bootstrap target B with batch_001 already ingested -> version 0.
spark.sql("""
    CREATE TABLE occ_spark_target USING DELTA AS
    SELECT * FROM all_batches WHERE filename = 'batch_001.csv'
""")
spark.sql("SELECT filename, count(*) AS rows FROM occ_spark_target GROUP BY filename").show()

StatementMeta(, 559211ba-4ce9-481e-9ff9-a90290762005, 3, Finished, Available, Finished, False)

+-------------+----+
|     filename|rows|
+-------------+----+
|batch_001.csv|  20|
+-------------+----+



## Step 1 — read B the normal way, decide the work list

The pipeline reads what is already in B and builds its work list from that: "ingest the batches B hasn't seen." This is the ordinary read every ETL job does, and it holds the result in the program. B has only `batch_001`, so the work list is `batch_002..005`.

In [2]:
# Ordinary read of the target, held in the program (what every pipeline does).
seen = [r.filename for r in
        spark.sql("SELECT DISTINCT filename FROM occ_spark_target").collect()]
print("B currently contains:", sorted(seen))

# Work list derived from what we just read. The NOT IN list IS the state we observed.
seen_sql = ", ".join(f"'{f}'" for f in seen)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW to_ingest AS
    SELECT * FROM all_batches WHERE filename NOT IN ({seen_sql})
""")
spark.sql("SELECT DISTINCT filename FROM to_ingest ORDER BY filename").show()

StatementMeta(, 559211ba-4ce9-481e-9ff9-a90290762005, 4, Finished, Available, Finished, False)

B currently contains: ['batch_001.csv']
+-------------+
|     filename|
+-------------+
|batch_002.csv|
|batch_003.csv|
|batch_004.csv|
|batch_005.csv|
+-------------+



## Step 2 — another process changes B (after our read, before our merge)

A concurrent job deletes `batch_001`. It lands after our read and before our merge — the realistic ETL gap. It is **not** concurrent with the merge's own transaction, which is exactly why Spark will not flag it.

In [3]:
spark.sql("DELETE FROM occ_spark_target WHERE filename = 'batch_001.csv'")
spark.sql("SELECT count(*) AS rows_in_B FROM occ_spark_target").show()
# B is now empty. Our work list still treats batch_001 as 'already ingested' and skips it.

StatementMeta(, 559211ba-4ce9-481e-9ff9-a90290762005, 5, Finished, Available, Finished, False)

+---------+
|rows_in_B|
+---------+
|        0|
+---------+



## Step 3 — merge the work list. Spark commits, no error.

The merge target is the live (now empty) table; the source is our work list `to_ingest`, decided from the pre-delete read. Spark opens its transaction at current HEAD, never learns the read it was based on is stale, and commits.

In [4]:
spark.sql("""
    MERGE INTO occ_spark_target AS t
    USING to_ingest AS s
    ON t.filename = s.filename
    WHEN NOT MATCHED THEN INSERT *
""")
print("MERGE committed — Spark raised no concurrency error")

StatementMeta(, 559211ba-4ce9-481e-9ff9-a90290762005, 6, Finished, Available, Finished, False)

MERGE committed — Spark raised no concurrency error


In [6]:
spark.sql("SELECT filename, count(*) AS rows FROM occ_spark_target GROUP BY filename ORDER BY filename").show()
spark.sql(""" select * FROM occ_spark_target WHERE filename = 'batch_001.csv' """).show()
spark.sql("DESCRIBE HISTORY occ_spark_target").select("version", "operation").show(truncate=False)

StatementMeta(, 559211ba-4ce9-481e-9ff9-a90290762005, 8, Finished, Available, Finished, False)

+-------------+----+
|     filename|rows|
+-------------+----+
|batch_002.csv|  20|
|batch_003.csv|  20|
|batch_004.csv|  20|
|batch_005.csv|  20|
+-------------+----+

+---+-----+--------+
| id|value|filename|
+---+-----+--------+
+---+-----+--------+

+-------+----------------------+
|version|operation             |
+-------+----------------------+
|2      |MERGE                 |
|1      |DELETE                |
|0      |CREATE TABLE AS SELECT|
+-------+----------------------+



## What happened

- B ends with `batch_002..005` and **no `batch_001`**. The delete won; the merge — whose work list was decided when `batch_001` was still in B — never re-inserted it. Data silently dropped, clean exit.
- A work list decided against HEAD (empty B) would have included `batch_001`. Ours was wrong the instant the delete landed, and Spark had no way to connect the two.

**Why Spark didn't catch it**

- The merge fixes one snapshot at transaction start (current HEAD = post-delete) and uses it for both its scan and its conflict check. Internally consistent — but bound to HEAD-at-merge-start, which Spark chose, not to the state our read saw.
- The delete committed *before* the merge transaction started, so the conflict checker sees no concurrent commit at all. Nothing to flag.
- Spark exposes no way to bind a merge to a version you read earlier. Reading the table natively leaves Spark with no memory of that read when the later merge runs.